<a href="https://colab.research.google.com/github/aditya-kumar-seth/cv/blob/main/digit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Handwritten Digit Recognition (Improved High Accuracy CNN)
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# ============================================================
# 1. LOAD DATA
# ============================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

labels = train["label"]
X = train.drop("label", axis=1)

# ============================================================
# 2. NORMALIZE PIXEL VALUES
# ============================================================

X = X / 255.0
test = test / 255.0

# ============================================================
# 3. RESHAPE DATA INTO IMAGE FORMAT
# ============================================================

X = X.values.reshape(-1, 28, 28, 1)
test = test.values.reshape(-1, 28, 28, 1)

# ============================================================
# 4. ONE HOT ENCODING
# ============================================================

Y = to_categorical(labels, 10)

# ============================================================
# 5. STRATIFIED TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_val, Y_train, Y_val = train_test_split(
    X,
    Y,
    test_size=0.1,
    random_state=42,
    stratify=labels
)

# ============================================================
# 6. DATA AUGMENTATION (OPTIMAL FOR MNIST)
# ============================================================

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10
)

datagen.fit(X_train)

# ============================================================
# 7. CNN MODEL ARCHITECTURE
# ============================================================

model = Sequential()

# -------- Block 1 --------
model.add(Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)))
model.add(BatchNormalization())

model.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())

model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))


# -------- Block 2 --------
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())

model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())

model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))


# -------- Block 3 --------
model.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())

model.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())

model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.30))


# -------- Fully Connected Layers --------
model.add(Flatten())

model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.50))

model.add(Dense(10, activation='softmax'))

# ============================================================
# 8. COMPILE MODEL
# ============================================================

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ============================================================
# 9. TRAINING CALLBACKS
# ============================================================

lr_scheduler = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=3,
    factor=0.5,
    min_lr=1e-5,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True
)

# ============================================================
# 10. TRAIN MODEL
# ============================================================

epochs = 50
batch_size = 64

history = model.fit(
    datagen.flow(X_train, Y_train, batch_size=batch_size),
    epochs=epochs,
    validation_data=(X_val, Y_val),
    steps_per_epoch=len(X_train) // batch_size,
    callbacks=[lr_scheduler, early_stop]
)

# ============================================================
# 11. TEST TIME AUGMENTATION (TTA)
# Improves final prediction accuracy
# ============================================================

tta_predictions = []

for i in range(5):

    preds = model.predict(
        datagen.flow(test, batch_size=128, shuffle=False)
    )

    tta_predictions.append(preds)

# Average predictions
predictions = np.mean(tta_predictions, axis=0)

predicted_labels = np.argmax(predictions, axis=1)

# ============================================================
# 12. CREATE SUBMISSION FILE
# ============================================================

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(predicted_labels) + 1),
    "Label": predicted_labels
})

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_11 (Conv2D)              │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 7, 7, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 586,986 (2.24 MB)

 Trainable params: 585,578 (2.23 MB)

...

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


590/590 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.7306 - loss: 0.9058 - val_accuracy: 0.9650 - val_loss: 0.1196 - learning_rate: 0.0010
Epoch 2/50
  1/590 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9688 - loss: 0.1541

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


590/590 ━━━━━━━━━━━━━━━━━━━━ 1s 826us/step - accuracy: 0.9688 - loss: 0.1541 - val_accuracy: 0.9664 - val_loss: 0.1130 - learning_rate: 0.0010
Epoch 3/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.9631 - loss: 0.1224 - val_accuracy: 0.9848 - val_loss: 0.0474 - learning_rate: 0.0010
Epoch 4/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 0s 778us/step - accuracy: 0.9531 - loss: 0.1025 - val_accuracy: 0.9824 - val_loss: 0.0491 - learning_rate: 0.0010
Epoch 5/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.9730 - loss: 0.0869 - val_accuracy: 0.9850 - val_loss: 0.0552 - learning_rate: 0.0010
Epoch 6/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 0s 790us/step - accuracy: 0.9844 - loss: 0.0515 - val_accuracy: 0.9855 - val_loss: 0.0555 - learning_rate: 0.0010
Epoch 7/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.9773 - loss: 0.0701 - val_accuracy: 0.9895 - val_loss: 0.0356 - learning_rate: 0.0010
Epoch 8/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9844 - loss: 0.0995 -

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# ============================================================
# Reproducibility
# ============================================================

SEED_VALUE = 123
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

# ============================================================
# Load CSV files
# ============================================================

print("Loading dataset...")

train_table = pd.read_csv("train.csv")
test_table  = pd.read_csv("test.csv")

digit_targets = train_table["label"].values
pixel_matrix  = train_table.drop("label", axis=1).values.astype("float32")

test_matrix = test_table.values.astype("float32")

# ============================================================
# Normalization
# ============================================================

pixel_matrix /= 255.0
test_matrix  /= 255.0

pixel_matrix = pixel_matrix.reshape(-1,28,28,1)
test_matrix  = test_matrix.reshape(-1,28,28,1)

encoded_targets = keras.utils.to_categorical(digit_targets,10)

# ============================================================
# Train validation split
# ============================================================

train_images, valid_images, train_labels, valid_labels = train_test_split(
    pixel_matrix,
    encoded_targets,
    test_size=0.1,
    random_state=SEED_VALUE,
    stratify=digit_targets
)

print("Train size:",train_images.shape[0])
print("Validation size:",valid_images.shape[0])

# ============================================================
# Data Augmentation
# ============================================================

image_transformer = ImageDataGenerator(
    rotation_range=12,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

image_transformer.fit(train_images)

# ============================================================
# Residual Unit
# ============================================================

def residual_layer(input_layer, filter_count):

    shortcut_connection = input_layer

    y = layers.Conv2D(filter_count,3,padding="same")(input_layer)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)

    y = layers.Conv2D(filter_count,3,padding="same")(y)
    y = layers.BatchNormalization()(y)

    y = layers.Add()([y,shortcut_connection])
    y = layers.Activation("relu")(y)

    return y

# ============================================================
# Build CNN Model
# ============================================================

def assemble_network():

    input_tensor = keras.Input(shape=(28,28,1))

    z = layers.Conv2D(32,3,padding="same",activation="relu")(input_tensor)
    z = layers.BatchNormalization()(z)

    z = residual_layer(z,32)
    z = layers.MaxPooling2D()(z)
    z = layers.Dropout(0.25)(z)

    z = layers.Conv2D(64,3,padding="same",activation="relu")(z)
    z = layers.BatchNormalization()(z)

    z = residual_layer(z,64)
    z = layers.MaxPooling2D()(z)
    z = layers.Dropout(0.30)(z)

    z = layers.Conv2D(128,3,padding="same",activation="relu")(z)
    z = layers.BatchNormalization()(z)

    z = residual_layer(z,128)
    z = layers.MaxPooling2D()(z)
    z = layers.Dropout(0.35)(z)

    z = layers.GlobalAveragePooling2D()(z)

    z = layers.Dense(128,activation="relu")(z)
    z = layers.BatchNormalization()(z)
    z = layers.Dropout(0.5)(z)

    output_tensor = layers.Dense(10,activation="softmax")(z)

    network = keras.Model(input_tensor,output_tensor)

    network.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return network

digit_model = assemble_network()
digit_model.summary()

# ============================================================
# Training callbacks
# ============================================================

callback_list = [

    ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_accuracy",
        patience=12,
        restore_best_weights=True,
        verbose=1
    )
]

# ============================================================
# Training phase
# ============================================================

MAX_EPOCHS = 10
BATCH_DIM  = 64

print("Starting training...")

training_record = digit_model.fit(
    image_transformer.flow(train_images,train_labels,batch_size=BATCH_DIM),
    epochs=MAX_EPOCHS,
    validation_data=(valid_images,valid_labels),
    callbacks=callback_list
)

# ============================================================
# Evaluate validation performance
# ============================================================

loss_value, accuracy_value = digit_model.evaluate(valid_images,valid_labels,verbose=0)

print("Validation accuracy:",accuracy_value)

# ============================================================
# Test Time Augmentation
# ============================================================

print("Running TTA inference...")

prediction_pool = []

for iteration in range(5):

    temp_preds = digit_model.predict(
        image_transformer.flow(test_matrix,batch_size=128,shuffle=False)
    )

    prediction_pool.append(temp_preds)

final_probs = np.mean(prediction_pool,axis=0)

final_digits = np.argmax(final_probs,axis=1)

# ============================================================
# Create submission file
# ============================================================

result_table = pd.DataFrame({

    "ImageId":np.arange(1,len(final_digits)+1),
    "Label":final_digits
})

result_table.to_csv("submission.csv",index=False)

print("submission.csv created")
print(result_table.head())

Loading dataset...
Train size: 37800
Validation size: 4200


Model: "functional_67"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 28, 28,    │        320 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 28, 28,    │      9,248 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 28, 28,    │      9,248 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_19[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 28, 28,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 28, 28,    │          0 │ add[0][0]         │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 14, 14,    │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 14, 14,    │          0 │ max_pooling2d_9[… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 14, 14,    │     18,496 │ dropout_12[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 14, 14,    │        256 │ conv2d_20[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 14, 14,    │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 14, 14,    │        256 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 14, 14,    │          0 │ batch_normalizat

 Total params: 501,194 (1.91 MB)

 Trainable params: 499,594 (1.91 MB)

 Non-trainable params: 1,600 (6.25 KB)

Starting training...
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


KeyboardInterrupt: 